### Mount Google Drive for persistent storage

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Check GPU availability and display GPU information

In [2]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Wed Mar 11 07:29:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [3]:
%%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Install system dependencies and clone the KenLM language model toolkit

In [4]:
!sudo apt-get update
!sudo apt-get install build-essential libboost-all-dev cmake zlib1g-dev libbz2-dev liblzma-dev
!sudo apt-get install libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm
%cd kenlm
!pip install e .

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,432 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,928 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-bac

### Build and install KenLM from source

In [5]:
!mkdir build
%cd build
!cmake ..
!make -j 4
!sudo make install

mkdir: cannot create directory ‘build’: File exists
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable v

### Add KenLM binaries to the system PATH and verify installation

In [6]:
import os
os.environ['PATH'] += ":/content/kenlm/build/bin"  # Use your exact path
!echo $PATH  # Verify
!which lmplz  # Test


/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/kenlm/build/bin
/usr/local/bin/lmplz


### Authenticate with Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


### Load the Dhivehi (Maldivian) Common Voice dataset (train+validation and test splits)

In [7]:
from datasets import load_dataset

common_voice_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
common_voice_test = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 100708.26it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 77181.45it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 93517.47it/s]


### Remove unused metadata columns (accent, age, client_id, etc.) from train and test sets and display random samples

In [8]:
common_voice_train = common_voice_train.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)
common_voice_test = common_voice_test.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)


,sentence,variant
0,މީގައި ގިނައީ ޓޫރިސްޓުން ކަމުގައި ވިޔަސް މާލޭގެ އިއްޒަތްތެރިންވެސް މަދެއް ނޫން,
1,ސަމާވެސް ޗުއްޓީގައި ރަށުން ބޭރުގައި ހުރުމުން ޒީނާއަށް ލިބުނީ އިތުރު އެކަނިވެރިކަމާއި ހިތްދަތި ކަމެއް,
2,ކުއްލި މަރުތަކުގެ ފަހަތުގައި ކޮން ކަމެއް؟,
3,އިސްލާމީ ޢިލްމާއި ދިވެހިބަހަށް އޮތް ލޯތްބެއްގެ ބޮޑުކަމުން,
4,ނުބައިކަންތަކާއި ހުތުރު ކަންތައް އެދުނިޔެއަކު ނުވެ,
5,ކޮޅުވެއްޓިއެއް ކުއްޔަށްހިފުމަށް,
6,ހުބަލު މަތިވެރިވެއްޖެ,
7,އަނެއް ކޮޓަރީގައި އުޅެނީ ރިސޯޓެއްގައި މަސައްކަތް ކުރާ މީހެއް,
8,ވައިފައްސިޔައަށް ދާއިރު ހުންގާނުގައި ހުންނަ މީހަކަށް ވެސް އެނގަހިފިދާނެ,
9,މިއަދާ ހަމައަށް ނުހުޅުވިވާ މަގޭ ލޯ ހުޅުވިއްޖެ,


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [9]:
import re

# remove punctuation etc.
chars_to_remove_regex = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, "", batch["sentence"]).lower()
    return batch

common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test = common_voice_test.map(remove_special_characters)

# remove rows that contain Arabic characters
arabic_pattern = re.compile(r"[\u0600-\u06FF]")

def remove_arabic_rows(batch):
    # True if NO Arabic characters → keep row
    return not bool(arabic_pattern.search(batch["sentence"]))

common_voice_train = common_voice_train.filter(remove_arabic_rows)
common_voice_test = common_voice_test.filter(remove_arabic_rows)

show_random_elements(common_voice_train.remove_columns(["path", "audio"]))


Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

,sentence,variant
0,އެސިފައިންގެ މީހަކީ މީހުނަށް ވަރަށް ހަމްދަރުދީވާ މީހެއް,
1,ލޯކަލް ކޮންސަލްޓަންޓް,
2,ކުރިން ބޭނުންކުރަމުން އައި މާނަ ފުޅާވެ ހަނިވެ,
3,އަހަރެންގެ ބައްޕައަކީ ހާދަ ނުބައި މީހެއް,
4,ޖެހެމުންދިޔައީ ތެތްފިނި ހިތްގައިމު ރޯޅިއެއް,
5,މަންމަ ހުރީ ވާނުވާ ނޭގިގެން ކަންތައް ބޮޑުވެފަ,
6,ހުރިހާ ހިއްސުތަކެއް,
7,ޓީޝާޓް މޫނުން ތިރިއަށް ފޭބިވަގުތު ބަނޑުން ޖަހާފައިވާ ބާލިދީގެ ދަށަށް ކަޅު ކުލައިގެ އެއްޗެއް ވަންހެން ޖަޒްލާއަށް ހީވި,
8,މިކަހަލަ ހިޔާލު ތަކެއްގައި އޮއްވައި އަހަރެންނަށް ނިދުން އިރެއްވެސް ނޭންގެ,
9,ލޮލުން ވެއްޓެންދިޔަ ކަރުނަތިކި ހިންދާލަމުން ޒާރާއާ ދިމާއަށް އެނބުރިލީ ހިނިތުންވެލަމުން,


### Extract the unique character vocabulary from train and test sets

In [10]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48, '–': 49, '’': 50, 'ﷲ': 51, 'ﷺ': 52}


### Remove any remaining Latin characters from transcriptions and re-extract the character vocabulary

In [11]:
def remove_latin_characters(batch):
    batch["sentence"] = re.sub(r"[a-z]+", "", batch["sentence"])
    return batch

# remove latin characters
common_voice_train = common_voice_train.map(remove_latin_characters)
common_voice_test = common_voice_test.map(remove_latin_characters)

# extract unique characters again
vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48, '–': 49, '’': 50, 'ﷲ': 51, 'ﷺ': 52}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [12]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 55


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [13]:
from transformers import Wav2Vec2CTCTokenizer
from transformers import SeamlessM4TFeatureExtractor
from transformers import Wav2Vec2BertProcessor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [14]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/e24cb6e0896730e3c68f4266eadc6dbf6150fbe071ba9968bf13481404180b10/dv_train_0/common_voice_dv_18580646.mp3', 'array': array([1.45519152e-11, 1.01863407e-10, 1.30967237e-10, ...,
       7.25846985e-06, 1.29183172e-06, 5.80571304e-06]), 'sampling_rate': 16000}
Target text: ޖެއްސުން ބޮޑު ސިޔާސަ ބައެއްގެ ތެރޭގައި އުޅޭނެ ގޮތް ވެސް ވަނީ ދަސްވެފަ
Input array shape: (101376,)
Sampling rate: 16000


### Write all transcriptions to a text file for KenLM language model training

In [15]:
with open("lm_corpus.txt", "w", encoding="utf-8") as f:
    for ex in common_voice_train:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")
    for ex in common_voice_test:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")

print("Wrote LM corpus to lm_corpus.txt")

Wrote LM corpus to lm_corpus.txt


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [16]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [17]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import evaluate

from transformers import Wav2Vec2BertProcessor

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

wer_metric = evaluate.load("wer")


### Train a trigram KenLM language model from the corpus

In [18]:
!lmplz -o 3 < lm_corpus.txt > lm.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /content/kenlm/build/lm_corpus.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 44800 types 17638
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:211656 2:15234823168 3:28565295104
Statistics:
1 17638 D1=0.74684 D2=1.12647 D3+=1.45216
2 41025 D1=0.916248 D2=1.20269 D3+=1.35257
3 43025 D1=0.96946 D2=1.40341 D3+=1.57432
Memory estimate for binary LM:
type      kB
probing 2165 assuming -p 1.5
probing 2474 assuming -r models -p 1.5
trie    1125 without quantization
trie     772 assuming -q 8 -b 8 quantization 
trie    1080 assuming -a 22 array pointer compression
trie     727 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:211656 2:656400 3:860500
----5---10---15

### Install pyctcdecode library for CTC beam search decoding with language model support

In [19]:
!pip install pyctcdecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.3/529.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 111.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requir

### Build a CTC beam search decoder with KenLM language model integration

In [20]:
from pyctcdecode import build_ctcdecoder

# get labels in ID order
vocab = processor.tokenizer.get_vocab()  # token -> id
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

# path to your trained KenLM model (arpa or binary)
kenlm_model_path = "lm.arpa"  # <-- change if you use another path/name

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,  # tune on dev
    beta=1.0   # tune on dev
)


### Define the evaluation metrics function using LM-aided beam search decoding to compute WER and CER

In [25]:
import evaluate
import torch

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    # Convert logits to probabilities
    probs = torch.softmax(torch.tensor(pred_logits), dim=-1).numpy()

    # Decode with LM
    pred_str = [decoder.decode(p, beam_width=64) for p in probs]

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [26]:
from transformers import Wav2Vec2BertForCTC

sinhala_checkpoint_dir = "/content/drive/MyDrive/final_model35"  # <-- wherever your Sinhala ./final_model is

model = Wav2Vec2BertForCTC.from_pretrained(
  sinhala_checkpoint_dir,              # <-- CHANGED (was facebook/w2v-bert-2.0)
  add_adapter=True,
  pad_token_id=processor.tokenizer.pad_token_id,
  vocab_size=len(processor.tokenizer), # keep Dhivehi vocab size here
  ignore_mismatched_sizes=True,        # <-- IMPORTANT for Sinhala->Dhivehi
)


Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at /content/drive/MyDrive/final_model35 and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([57]) in the model instantiated
- lm_head.weight: found shape torch.Size([80, 1024]) in the checkpoint and torch.Size([57, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [27]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=2700,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor
)


/tmp/ipykernel_1573/723348825.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [28]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 56, 'bos_token_id': 55}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2524.330200,220.008865,4.699643,2.109100
600,676.802600,185.396896,4.719808,2.111758


Step,Training Loss,Validation Loss,Wer,Cer
300,2524.330200,220.008865,4.699643,2.109100
600,676.802600,185.396896,4.719808,2.111758
900,543.129000,153.452255,4.690835,2.104288
1200,461.637000,149.727005,4.703938,2.106702


TrainOutput(global_step=1450, training_loss=952.3932543103448, metrics={'train_runtime': 29804.5392, 'train_samples_per_second': 1.556, 'train_steps_per_second': 0.049, 'total_flos': 7.189631826556394e+18, 'train_loss': 952.3932543103448, 'epoch': 10.0})

### Save the fine-tuned model and processor to disk

In [ ]:
trainer.save_model("./final_model")
processor.save_pretrained("./final_model")
print("Saved fine-tuned model + processor to ./final_model")


Saved fine-tuned model + processor to ./final_model


### Build a CTC beam search decoder with KenLM language model integration

In [ ]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

checkpoint_dir = "./"  # or some checkpoint in ./outputs

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)


vocab = processor.tokenizer.get_vocab()
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,
    beta=1.0,
)

example = common_voice_test[0]

with torch.no_grad():
    logits = model(
        torch.tensor(example["input_features"]).unsqueeze(0).to("cuda")
    ).logits

probs = torch.softmax(logits[0], dim=-1).cpu().numpy()
pred_text = decoder.decode(probs)

ref_text = processor.decode(example["labels"], group_tokens=False).lower()

print("Reference :", ref_text)
print("Prediction:", pred_text)


Reference : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
Prediction: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން


### Run inference on the test set using LM-aided beam search decoding and compute WER/CER

In [30]:
import numpy as np

model.eval()

all_pred_raw = []        # ASR + LM only
all_pred_corrected = []  # ASR + LM + T5 spell corrector
all_refs = []            # ground-truth text

for i, ex in enumerate(common_voice_test):
    # 1. Forward pass on stored input_features from your prepared dataset
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(input_feats).logits

    # 2. Convert logits to probabilities
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # 3. Decode with LM (wav2vec2 + CTC + KenLM)
    pred_text = decoder.decode(probs)
    all_pred_raw.append(pred_text)

    # 5. Reference text from labels
    ref_text = processor.decode(ex["labels"], group_tokens=False).lower()
    all_refs.append(ref_text)

# 6. Compute WERs
wer_raw = wer_metric.compute(
    predictions=all_pred_raw,
    references=all_refs,
)
cer_raw = cer_metric.compute(predictions=all_pred_raw, references=all_refs)

print(f"WER (ASR + LM only)           : {wer_raw:.4f}")
print(f"CER (ASR + LM only)           : {cer_raw:.4f}")

# Optional: inspect a few examples
for idx in range(3):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("RAW :", all_pred_raw[idx])

WER (ASR + LM only)           : 0.1515
CER (ASR + LM only)           : 0.0348

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
RAW : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
RAW : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
RAW : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
